### Connexion à la DB DuckDB

In [1]:
import duckdb
import os
from pathlib import Path
from typing import List
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
from tqdm import tqdm
import sklearn
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import plotly.express as px

### Connexion à la DB / Import des Data


In [2]:
# Store database at project root
DB_NAME = Path("/home/c-enjalbert/Documents/EPSI/MSPR/bloc_2/amazing/amazing.duckdb") 
# Go up one level from current directory to get to project root
data_folder = Path("..") / "data"
# For absolute certainty, you could use the absolute path
# data_folder = Path("/home/c-enjalbert/Documents/EPSI/MSPR/bloc_2/amazing/data")
con = duckdb.connect(str(DB_NAME))

In [3]:
# 2. Query to list all tables in the database
# DuckDB specific way to list tables
tables_info = con.sql("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
""").df()

print(f"Found {len(tables_info)} tables in the database:\n")

if len(tables_info) > 0:
    for i, table_name in enumerate(tables_info['table_name']):
        print(f"{i+1}. {table_name}")
else:
    print("No tables found in the database.")

Found 4 tables in the database:

1. all_events
2. loaded_files
3. user_events
4. user_segments_kmeans


### Import de la table DuckDB

In [4]:
DB_NAME = "amazing.duckdb"
TABLE_EVENTS = "user_events"
TABLE_SEGMENTS = "user_segments"
SAMPLE_USER_PERCENT = 0.005
BATCH_SIZE = 1000 

In [5]:
# 5. Alternative way to show all tables
print("List of all tables using DuckDB's connections.tables():")
con.sql("SHOW TABLES").show()

List of all tables using DuckDB's connections.tables():
┌──────────────────────┐
│         name         │
│       varchar        │
├──────────────────────┤
│ all_events           │
│ loaded_files         │
│ user_events          │
│ user_segments_kmeans │
└──────────────────────┘



In [6]:
# Examine the all_events table
print(f"First 10 rows of {TABLE_EVENTS} table:")
user_events = con.sql(f"""
    SELECT DISTINCT category_code FROM all_events
""")
user_events.show()
user_events_df = user_events.df()


First 10 rows of user_events table:
┌────────────────────────────────────────┐
│             category_code              │
│                varchar                 │
├────────────────────────────────────────┤
│ furniture.living_room.sofa             │
│ apparel.shirt                          │
│ appliances.kitchen.blender             │
│ appliances.kitchen.hob                 │
│ appliances.environment.air_conditioner │
│ apparel.skirt                          │
│ appliances.environment.water_heater    │
│ appliances.environment.fan             │
│ furniture.bedroom.bed                  │
│ accessories.bag                        │
│        ·                               │
│        ·                               │
│        ·                               │
│ electronics.audio.acoustic             │
│ appliances.personal.scales             │
│ computers.peripherals.camera           │
│ apparel.shoes                          │
│ sport.ski                              │
│ sport.trainer   

In [9]:
user_events_df.value_counts()

category_code       
accessories.bag         1
accessories.umbrella    1
accessories.wallet      1
apparel.belt            1
apparel.costume         1
                       ..
sport.ski               1
sport.snowboard         1
sport.tennis            1
sport.trainer           1
stationery.cartrige     1
Name: count, Length: 138, dtype: int64

In [ ]:
user_events_df.value_counts().count()

In [28]:
# Extract the first level of category (before the first dot)
user_events_df['category_family'] = user_events_df['category_code'].str.split('.').str[0]

# Display value counts grouped by first family
user_events_df['category_family'].value_counts()

category_family
appliances      30
apparel         24
computers       16
electronics     14
furniture       12
construction    11
auto             9
sport            6
kids             6
country_yard     5
accessories      3
stationery       1
medicine         1
Name: count, dtype: int64

In [29]:
# Count the number of distinct category families
user_events_df['category_family'].nunique()

13

In [30]:
# Extract the first level of category (before the first dot)
user_events_df['category_family'] = user_events_df['category_code'].str.split('.').str[0]

# Display value counts grouped by first family
counts = user_events_df['category_family'].value_counts()
print(f"Total number of category families: {len(counts)}\n")
counts

Total number of category families: 13



category_family
appliances      30
apparel         24
computers       16
electronics     14
furniture       12
construction    11
auto             9
sport            6
kids             6
country_yard     5
accessories      3
stationery       1
medicine         1
Name: count, dtype: int64

In [ ]:
user_events_df = user_events_df.dropna(subset=["category_embedding"])

In [ ]:
user_events_df["category_embedding"].value_counts()

np.int64(4905)

In [ ]:
user_events_df["category_embedding"].isna().value_counts()

category_embedding
False    4905
Name: count, dtype: int64

In [ ]:
user_events_df = user_events_df.dropna(subset=["category_embedding"])

In [ ]:
user_events_df["category_embedding"].isna().value_counts()

category_embedding
False    4905
Name: count, dtype: int64

## Fonctions

In [ ]:
def embed_plot_3d_embedding_only(category_embedding):
    # --- Visualisation des embeddings en 3D ---
    print("Visualisation des embeddings en 3D...")
    pca = PCA(n_components=3)
    X_pca = pca.fit_transform(category_embedding)

    # Prepare data for Plotly
    plot_data = pd.DataFrame({
        "SVD Component 1": X_pca[:, 0],
        "SVD Component 2": X_pca[:, 1],
        "SVD Component 3": X_pca[:, 2]
    })

    # Create an interactive 3D scatter plot
    fig = px.scatter_3d(
        plot_data,
        x="SVD Component 1",
        y="SVD Component 2",
        z="SVD Component 3",
        title="Interactive 3D Plot of Clusters",
        opacity=0.8
    )

    # Show the plot
    fig.show()

# Analyse des embeddings

In [ ]:
category_family_embedding = np.vstack(user_events_df["category_family_embedding"].values)

In [ ]:
category_embedding = np.vstack(user_events_df["category_embedding"].values)

In [ ]:
user_events_df["category_embedding"]

0       [0.35857014219189, 0.2146849604615403, 0.66510...
1       [0.541805149360225, 0.29873900777542917, 0.609...
2       [0.677823864283297, -0.19088278029106448, -0.0...
3       [0.36524954182900954, 0.4529579710213816, -0.1...
4       [0.23150635567652902, 0.24461066076096485, 0.4...
                              ...                        
4995    [0.223866140498235, 0.23939956580190141, -0.01...
4996    [0.8899611793437568, -0.35902182733139465, -0....
4997    [0.3904866375637094, 0.6501394777730585, -0.58...
4998    [0.8902528008836732, -0.4115924806048521, -0.0...
4999    [0.3627508150349448, 0.03740835381976232, 0.23...
Name: category_embedding, Length: 4905, dtype: object

In [ ]:
embed_plot_3d_embedding_only(category_embedding)

Visualisation des embeddings en 3D...
